# Fake BoTorch Optimization Debug Notebook

This notebook exercises the new generic online optimizer without HFSS/server dependencies. It uses a finite `DiscreteCandidateSpace` and a fake lookup simulator that only supports limited variable combinations.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from optimization.candidate_space import DiscreteCandidateSpace
from optimization.optimization_spec import BCLFOptimizationSpec
from optimization.optimizer_bo import OptimizerBoTorch
from optimization.recorder import ExperimentRecorder
from optimization.simulators import SimulationResult, Simulator


In [ ]:
class FakeBCLFSimulator(Simulator):
    def __init__(self, *, supported_geometries, freqs_hz):
        super().__init__()
        self.supported_geometries = np.asarray(supported_geometries, dtype=float)
        self.freqs = np.asarray(freqs_hz, dtype=float)

    def simulate(self, geometry):
        self.num_calls += 1
        requested = np.asarray(geometry, dtype=float).reshape(-1)
        dists = np.sum((self.supported_geometries - requested) ** 2, axis=1)
        actual = self.supported_geometries[int(np.argmin(dists))]
        freqs_ghz = self.freqs / 1e9

        # Synthetic RF response: geometry shifts center frequency and stopband leakage.
        center = 5.55 + 0.18 * (actual[0] - 1.0) - 0.10 * (actual[1] - 1.0)
        width = 0.30 + 0.04 * actual[2]
        leakage = -22.0 + 2.5 * abs(actual[0] - 1.0) + 1.5 * abs(actual[1] - 1.0)
        pass_shape = -1.5 - 18.0 * ((freqs_ghz - center) / width) ** 2
        s21_db = np.maximum(leakage, pass_shape)
        s11_db = -1.0 - 24.0 * np.exp(-((freqs_ghz - center) / 0.22) ** 2)

        s_params = np.zeros((2, 2, len(self.freqs)), dtype=complex)
        s_params[1, 0, :] = 10 ** (s21_db / 20.0)
        s_params[0, 0, :] = 10 ** (s11_db / 20.0)
        self.num_success += 1
        return SimulationResult(
            freqs=self.freqs,
            s_params=s_params,
            status='done',
            metadata={'cache_hit': False, 'case_id': f'fake_{self.num_calls:04d}'},
            actual_geometry=actual,
        )


In [ ]:
freqs_hz = np.linspace(3.5e9, 8.8e9, 401)
values = [0.0, 0.5, 1.0, 1.5, 2.0]
supported = np.asarray([[x, y, z] for x in values for y in values for z in values], dtype=float)

candidate_space = DiscreteCandidateSpace(
    variable_order=['x_shift', 'y_shift', 'width_code'],
    candidate_geometries=supported,
)
simulator = FakeBCLFSimulator(supported_geometries=supported, freqs_hz=freqs_hz)

history_snapshots = []
def live_callback(trial, history):
    history_snapshots.append(pd.DataFrame(history))

optimizer = OptimizerBoTorch(
    candidate_space=candidate_space,
    simulator=simulator,
    optimization_spec=BCLFOptimizationSpec(),
    recorder=ExperimentRecorder(output_dir=ROOT / 'experiments' / 'bo_fake_botorch'),
    budget=25,
    n_init=5,
    seed=0,
    run_id='bo_fake_botorch_seed0',
    checkpoint_path=ROOT / 'experiments' / 'bo_fake_botorch' / 'bo_fake_botorch_seed0.checkpoint.json',
    progress_callback=live_callback,
)
result = optimizer.run()
print('best objective:', result.state.best_objective)
print('best geometry:', result.state.best_geometry)
print('json:', result.recorder_output.json_path)


In [ ]:
df = pd.DataFrame(result.history)
display(df[['iteration', 'phase', 'objective', 'best_so_far', 'requested_geometry', 'geometry', 'geometry_normalized', 'acquisition_score']].tail(15))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df['iteration'], df['objective'], marker='o')
axes[0].set_title('Objective by iteration')
axes[0].set_xlabel('iteration')
axes[0].set_ylabel('objective (minimize)')
axes[0].grid(True, alpha=0.3)
axes[1].plot(df['iteration'], df['best_so_far'], marker='s')
axes[1].set_title('Best-so-far')
axes[1].set_xlabel('iteration')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()


In [ ]:
best_geo = np.asarray(result.best_trial['geometry'], dtype=float)
best_sim = simulator.simulate(best_geo)
freqs_ghz = best_sim.freqs / 1e9
s21_db = 20 * np.log10(np.maximum(np.abs(best_sim.s_params[1, 0, :]), 1e-15))
s11_db = 20 * np.log10(np.maximum(np.abs(best_sim.s_params[0, 0, :]), 1e-15))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax in axes:
    ax.axvspan(3.6, 4.8, color='tab:red', alpha=0.12, label='low stopband')
    ax.axvspan(5.2, 5.9, color='tab:green', alpha=0.12, label='passband')
    ax.axvspan(6.5, 8.7, color='tab:orange', alpha=0.12, label='high stopband')
    ax.grid(True, alpha=0.3)
axes[0].plot(freqs_ghz, s21_db, label='S21')
axes[0].set_title('Best fake S21')
axes[0].set_xlabel('Frequency (GHz)')
axes[0].set_ylabel('dB')
axes[0].legend()
axes[1].plot(freqs_ghz, s11_db, label='S11')
axes[1].set_title('Best fake S11')
axes[1].set_xlabel('Frequency (GHz)')
axes[1].set_ylabel('dB')
axes[1].legend()
plt.tight_layout()
